In [ ]:
import ftplib
import os
from pathlib import Path

HOST = 'ftpupload.net'
USER = 'if0_41831439'
PASSWD = 'FUFioL02wkCvR'
REMOTE_ROOT = 'htdocs'
LOCAL_ROOT = Path('dist/public')

print('Local root:', LOCAL_ROOT.resolve())
if not LOCAL_ROOT.exists():
    raise FileNotFoundError(f'Local build directory not found: {LOCAL_ROOT}')

ftp = ftplib.FTP(HOST, timeout=30)
ftp.login(USER, PASSWD)
print('Connected:', ftp.getwelcome())
ftp.cwd(REMOTE_ROOT)

def ensure_remote_dir(path):
    if not path:
        return
    parts = path.split('/')
    current = []
    for part in parts:
        if not part:
            continue
        current.append(part)
        folder = '/'.join(current)
        try:
            ftp.mkd(folder)
            print('Created remote dir:', folder)
        except ftplib.error_perm as err:
            if '550' in str(err):
                pass
            else:
                raise

def upload_directory(local_dir: Path, remote_dir=''):
    for entry in sorted(local_dir.iterdir()):
        remote_path = f'{remote_dir}/{entry.name}' if remote_dir else entry.name
        if entry.is_dir():
            ensure_remote_dir(remote_path)
            upload_directory(entry, remote_path)
        else:
            print('Uploading', remote_path)
            with entry.open('rb') as f:
                ftp.storbinary(f'STOR {remote_path}', f)

upload_directory(LOCAL_ROOT)
ftp.quit()
print('FTP upload complete')